In [1]:
from __future__ import annotations

import os
import sys
import re
import random
from pathlib import Path
from dataclasses import dataclass
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from benchmarks.hierarchical_scoring import (
    _expected_major_state_generic,
    _parse_state_generic,
)
from benchmarks.cd8_config import CD8_HIER_CFG
from benchmarks.cd4_config import CD4_HIER_CFG
from benchmarks.caf_config import CAF_HIER_CFG
from benchmarks.mouse_b_config import MOUSE_B_CFG

warnings.filterwarnings("ignore")

In [2]:
# =============================================================================
# 0. Reproducibility / paths
# =============================================================================
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

BASE = Path("/work")
INPUT_DIR = BASE / "paper" / "gb_resubmission" / "input"
OUTPUT_DIR = BASE / "paper" / "gb_resubmission" / "output" / "Fig3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("OUTPUT_DIR:", OUTPUT_DIR)


OUTPUT_DIR: /work/paper/gb_resubmission/output/Fig3


In [3]:
# ------------------------------------------------------------
# 1. Dataset / pipeline config
# ------------------------------------------------------------
PIPELINE_ORDER = ["Standard", "Curated", "CellTypist", "SingleR", "Azimuth"]

PIPELINE_LABELS = {
    "Standard":   "Standard\n(top DEGs)",
    "Curated":    "Full pipeline\n(LLM-scCurator)",
    "CellTypist": "CellTypist",
    "SingleR":    "SingleR",
    "Azimuth":    "Azimuth",
}

PIPELINE_COLORS = {
    "Standard":   "#42A5F5",
    "Curated":    "#D32F2F",
    "CellTypist": "#66BB6A",
    "SingleR":    "#7E57C2",
    "Azimuth":    "#FFB74D",
}

DATASET_ORDER = ["CD8", "CD4", "MSC", "MOUSE_B"]

DATASET_SPECS = {
    "CD8": {
        "csv": os.path.join(INPUT_DIR, "cd8_benchmark_results_integrated_SCORED.csv"),
        "cfg": CD8_HIER_CFG,
        "display": "CD8",
        "state_to_major": {
            "Naive": "T_cell",
            "EffMem": "T_cell",
            "Exhausted": "T_cell",
            "Resident": "T_cell",
            "MAIT": "T_cell",
            "ISG": "T_cell",
            "Cycling": "T_cell",
            "Other": "Other",
        },
    },
    "CD4": {
        "csv": os.path.join(INPUT_DIR, "cd4_benchmark_results_integrated_SCORED.csv"),
        "cfg": CD4_HIER_CFG,
        "display": "CD4",
        "state_to_major": {
            "Naive": "T_cell",
            "EffMem": "T_cell",
            "Exhausted": "T_cell",
            "Treg": "T_cell",
            "Tfh": "T_cell",
            "Th17": "T_cell",
            "ISG": "T_cell",
            "Cycling": "T_cell",
            "Other": "Other",
        },
    },
    "MSC": {
        "csv": os.path.join(INPUT_DIR, "msc_benchmark_results_integrated_SCORED.csv"),
        "cfg": CAF_HIER_CFG,
        "display": "MSC",
        "state_to_major": {
            "iCAF": "Fibroblast",
            "myCAF": "Fibroblast",
            "PVL": "Fibroblast",
            "Cycling": "Fibroblast",
            "Endothelial": "Endothelial",
            "Other": "Other",
        },
    },
    "MOUSE_B": {
        "csv": os.path.join(INPUT_DIR, "mouse_b_benchmark_results_integrated_SCORED.csv"),
        "cfg": MOUSE_B_CFG,
        "display": "MOUSE_B",
        "state_to_major": {
            "Mature_B": "B_lineage",
            "Erythrocyte_like": "Erythroid",
            "Mast_like": "Mast",
            "pDC_Myeloid_like": "pDC_Myeloid",
            "Other": "Other",
        },
    },
}

In [4]:
# ------------------------------------------------------------
# 2. Plot style
# ------------------------------------------------------------
@dataclass(frozen=True)
class FigSpec:
    PAD_IN: float = 0.03
    DPI_PNG: int = 600

def _set_gb_rc(base_pt: float = 9.0) -> None:
    mpl.rcParams.update({
        "font.size": base_pt,
        "axes.labelsize": base_pt,
        "axes.titlesize": base_pt,
        "xtick.labelsize": base_pt - 0.8,
        "ytick.labelsize": base_pt - 0.8,
        "legend.fontsize": base_pt - 0.2,
        "axes.linewidth": 0.6,
        "xtick.major.width": 0.6,
        "ytick.major.width": 0.6,
        "xtick.major.size": 2.5,
        "ytick.major.size": 2.5,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans"],
    })

def _save(fig, out_pdf: str, dpi_png: int = 600, pad_inches: float = 0.03):
    fig.savefig(out_pdf, bbox_inches="tight", pad_inches=pad_inches)
    fig.savefig(str(Path(out_pdf).with_suffix(".png")), dpi=dpi_png, bbox_inches="tight", pad_inches=pad_inches)

In [5]:
# ------------------------------------------------------------
# 3. Helpers
# ------------------------------------------------------------
def choose_prediction_column(df: pd.DataFrame, pipeline: str) -> str:
    """
    Prefer cleaned subtype columns for evaluation if available.
    Otherwise fall back to the original answer column.
    """
    preferred = [
        f"{pipeline}_Subtype_Clean",
        f"{pipeline}_Answer",
    ]
    for col in preferred:
        if col in df.columns:
            return col
    raise ValueError(f"No prediction column found for pipeline={pipeline}")

def bootstrap_mean_ci(values, n_boot: int = 5000, seed: int = 42, alpha: float = 0.05):
    arr = pd.Series(values).dropna().astype(float).values
    if len(arr) == 0:
        return np.nan, np.nan, np.nan
    mean = float(arr.mean())
    if len(arr) == 1:
        return mean, mean, mean

    rng = np.random.default_rng(seed)
    boots = np.empty(n_boot, dtype=float)
    n = len(arr)
    for i in range(n_boot):
        sample = rng.choice(arr, size=n, replace=True)
        boots[i] = sample.mean()

    lo = float(np.quantile(boots, alpha / 2))
    hi = float(np.quantile(boots, 1 - alpha / 2))
    return mean, lo, hi

def canonical_major_from_state(state: str, state_to_major: dict) -> str:
    return state_to_major.get(str(state), "Other")

def parse_major_from_text_fallback(text: str, dataset_name: str) -> str:
    """
    Fallback major-lineage parser from raw text.
    Used only when state parsing lands in 'Other' or is uninformative.
    """
    t = str(text).strip().lower()

    if dataset_name in {"CD8", "CD4"}:
        t_markers = [
            "t cell", "t-cell", "cd8", "cd4", "naive", "effector", "memory",
            "exhaust", "resident", "mait", "treg", "tfh", "th17", "isg"
        ]
        if any(k in t for k in t_markers):
            return "T_cell"
        return "Other"

    if dataset_name == "MSC":
        if any(k in t for k in ["endothelial", "pecam1", "plvap", "vwf"]):
            return "Endothelial"
        if any(k in t for k in ["fibro", "caf", "icaf", "mycaf", "perivascular", "pvl"]):
            return "Fibroblast"
        return "Other"

    if dataset_name == "MOUSE_B":
        if any(k in t for k in ["b cell", "b-cell", "mature b", "cd79", "ms4a1"]):
            return "B_lineage"
        if any(k in t for k in ["eryth", "hemoglobin", "hba", "gypa"]):
            return "Erythroid"
        if any(k in t for k in ["mast", "mcpt", "cpa3"]):
            return "Mast"
        if any(k in t for k in ["pdc", "plasmacytoid", "myeloid", "lyz"]):
            return "pDC_Myeloid"
        return "Other"

    return "Other"

def add_gt_fields(df: pd.DataFrame, cfg, state_to_major: dict) -> pd.DataFrame:
    """
    Adds GT_Major / GT_State / UsedInConfusion.

    GT_Major is canonicalized from GT_State using the same state_to_major map
    used on the prediction side, so both sides share the same label space.
    """
    df = df.copy()

    gt_major_list = []
    gt_state_list = []
    used_list = []

    for gt in df["Ground_Truth"].astype(str):
        _, state = _expected_major_state_generic(gt, cfg)
        gt_state = str(state)
        gt_major = canonical_major_from_state(gt_state, state_to_major)

        gt_major_list.append(gt_major)
        gt_state_list.append(gt_state)
        used_list.append(gt_state != cfg.default_state)

    df["GT_Major"] = gt_major_list
    df["GT_State"] = gt_state_list
    df["UsedInConfusion"] = used_list
    return df

def parse_pred_state_and_major(text: str, cfg, state_to_major: dict, dataset_name: str):
    """
    Parse predicted state, then canonicalize major lineage.
    If state parsing falls to 'Other', use a raw-text fallback.
    """
    raw_text = str(text)
    pred_state = _parse_state_generic(raw_text, cfg)
    pred_major = canonical_major_from_state(pred_state, state_to_major)

    if pred_major == "Other":
        pred_major = parse_major_from_text_fallback(raw_text, dataset_name)

    return pred_state, pred_major

def compute_dataset_pipeline_metrics(
    df: pd.DataFrame,
    dataset_name: str,
    pipeline: str,
    cfg,
    state_to_major: dict,
) -> dict | None:
    score_col = f"Score_{pipeline}"
    if score_col not in df.columns:
        return None

    pred_col = choose_prediction_column(df, pipeline)

    # Primary benchmark metric
    sanno_scores = df[score_col].dropna().astype(float)
    mean_sanno, ci_lo, ci_hi = bootstrap_mean_ci(sanno_scores.values, n_boot=5000, seed=42)

    # Ontology-aware helper metrics derived directly from Sanno
    frac_eq1 = float((sanno_scores == 1.0).mean()) if len(sanno_scores) > 0 else np.nan
    frac_ge05 = float((sanno_scores >= 0.5).mean()) if len(sanno_scores) > 0 else np.nan

    # Diagnostic exact-match / major-lineage metrics
    eval_df = df[df["UsedInConfusion"]].copy()
    if eval_df.empty:
        exact_acc = np.nan
        major_acc = np.nan
        n_eval = 0
        pred_state_other_rate = np.nan
    else:
        pred_states = []
        pred_majors = []

        for txt in eval_df[pred_col].fillna("Unknown").astype(str):
            ps, pm = parse_pred_state_and_major(txt, cfg, state_to_major, dataset_name)
            pred_states.append(ps)
            pred_majors.append(pm)

        eval_df["Pred_State"] = pred_states
        eval_df["Pred_Major"] = pred_majors

        exact_acc = float((eval_df["Pred_State"] == eval_df["GT_State"]).mean())
        major_acc = float((eval_df["Pred_Major"] == eval_df["GT_Major"]).mean())
        n_eval = int(len(eval_df))
        pred_state_other_rate = float((eval_df["Pred_State"] == "Other").mean())

    return {
        "Dataset": dataset_name,
        "Pipeline": pipeline,
        "PredictionColumn": pred_col,
        "N_clusters": int(len(sanno_scores)),
        "N_eval": n_eval,
        "MeanSanno": mean_sanno,
        "MeanSanno_CI_Low": ci_lo,
        "MeanSanno_CI_High": ci_hi,
        "MeanSanno_pct": mean_sanno * 100 if pd.notna(mean_sanno) else np.nan,
        "MeanSanno_CI_Low_pct": ci_lo * 100 if pd.notna(ci_lo) else np.nan,
        "MeanSanno_CI_High_pct": ci_hi * 100 if pd.notna(ci_hi) else np.nan,
        "FracScoreEq1": frac_eq1,
        "FracScoreEq1_pct": frac_eq1 * 100 if pd.notna(frac_eq1) else np.nan,
        "FracScoreGe0_5": frac_ge05,
        "FracScoreGe0_5_pct": frac_ge05 * 100 if pd.notna(frac_ge05) else np.nan,
        "ExactMatch": exact_acc,
        "ExactMatch_pct": exact_acc * 100 if pd.notna(exact_acc) else np.nan,
        "MajorLineageAcc": major_acc,
        "MajorLineageAcc_pct": major_acc * 100 if pd.notna(major_acc) else np.nan,
        "PredState_Other_Rate": pred_state_other_rate,
        "PredState_Other_Rate_pct": pred_state_other_rate * 100 if pd.notna(pred_state_other_rate) else np.nan,
    }

def build_fig3_summary():
    rows = []

    for ds_name in DATASET_ORDER:
        spec = DATASET_SPECS[ds_name]
        csv_path = spec["csv"]

        if not os.path.exists(csv_path):
            print(f"[WARN] missing: {csv_path}")
            continue

        df = pd.read_csv(csv_path)
        df = add_gt_fields(df, spec["cfg"], spec["state_to_major"])

        for pipeline in PIPELINE_ORDER:
            res = compute_dataset_pipeline_metrics(
                df=df,
                dataset_name=ds_name,
                pipeline=pipeline,
                cfg=spec["cfg"],
                state_to_major=spec["state_to_major"],
            )
            if res is not None:
                rows.append(res)

    summary_df = pd.DataFrame(rows)
    return summary_df

def print_debug_tables():
    """
    Sanity check before plotting.
    """
    print("\n================ DEBUG SUMMARY ================\n")
    for ds_name in DATASET_ORDER:
        spec = DATASET_SPECS[ds_name]
        if not os.path.exists(spec["csv"]):
            continue

        df = pd.read_csv(spec["csv"])
        df = add_gt_fields(df, spec["cfg"], spec["state_to_major"])

        print(f"\n--- {ds_name} ---")
        for pipeline in PIPELINE_ORDER:
            score_col = f"Score_{pipeline}"
            if score_col not in df.columns:
                continue

            pred_col = choose_prediction_column(df, pipeline)
            sub = df[df["UsedInConfusion"]].copy()

            pred_states = []
            pred_majors = []
            for txt in sub[pred_col].fillna("Unknown").astype(str):
                ps, pm = parse_pred_state_and_major(txt, spec["cfg"], spec["state_to_major"], ds_name)
                pred_states.append(ps)
                pred_majors.append(pm)

            sub["Pred_State"] = pred_states
            sub["Pred_Major"] = pred_majors

            score_vals = sub[score_col].dropna().astype(float)

            exact_val = 100.0 * (sub["Pred_State"] == sub["GT_State"]).mean() if len(sub) > 0 else np.nan
            major_val = 100.0 * (sub["Pred_Major"] == sub["GT_Major"]).mean() if len(sub) > 0 else np.nan
            other_val = 100.0 * (sub["Pred_State"] == "Other").mean() if len(sub) > 0 else np.nan
            mean_sanno = 100.0 * score_vals.mean() if len(score_vals) > 0 else np.nan
            frac_eq1 = 100.0 * (score_vals == 1.0).mean() if len(score_vals) > 0 else np.nan
            frac_ge05 = 100.0 * (score_vals >= 0.5).mean() if len(score_vals) > 0 else np.nan

            print(
                f"{pipeline:10s} | pred_col={pred_col:20s} | "
                f"MeanSanno={mean_sanno:5.1f}% | "
                f"Sanno==1={frac_eq1:5.1f}% | "
                f"Sanno>=0.5={frac_ge05:5.1f}% | "
                f"Exact={exact_val:5.1f}% | "
                f"Major={major_val:5.1f}% | "
                f"PredStateOther={other_val:5.1f}%"
            )


In [6]:
# ------------------------------------------------------------
# 4. Heatmap helper
# ------------------------------------------------------------
def draw_metric_heatmap(
    ax,
    summary_df: pd.DataFrame,
    value_col: str,
    title: str,
    vmin: float = 0.0,
    vmax: float = 100.0,
    cmap: str = "YlGnBu",
    show_ylabels: bool = True,
):
    mat = (
        summary_df
        .pivot(index="Dataset", columns="Pipeline", values=value_col)
        .reindex(index=DATASET_ORDER, columns=PIPELINE_ORDER)
    )

    im = ax.imshow(mat.values, aspect="auto", vmin=vmin, vmax=vmax, cmap=cmap)

    ax.set_xticks(np.arange(len(PIPELINE_ORDER)))
    ax.set_yticks(np.arange(len(DATASET_ORDER)))

    ax.set_xticklabels(
        [PIPELINE_LABELS[p] for p in PIPELINE_ORDER],
        rotation=28,
        ha="right"
    )

    if show_ylabels:
        ax.set_yticklabels(DATASET_ORDER)
    else:
        ax.set_yticklabels([])
        ax.tick_params(axis="y", length=0)

    #ax.set_title(title, fontweight="bold", pad=6)
    ax.set_xlabel(title, fontweight="bold")

    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat.iloc[i, j]
            if pd.isna(val):
                ax.text(j, i, "NA", ha="center", va="center", fontsize=8, color="black")
            else:
                txt_color = "white" if val >= 60 else "black"
                ax.text(j, i, f"{val:.1f}", ha="center", va="center", fontsize=8, color=txt_color)

    return im


In [7]:
# ------------------------------------------------------------
# 5. Main Fig. 3
# ------------------------------------------------------------
def plot_fig3_main(summary_df: pd.DataFrame, outdir: str):
    """
    Main Fig. 3:
      a, Mean Sanno (%)
      b, Ontology-consistent accuracy (%)  [Sanno >= 0.5]
      c, Major-lineage accuracy (%)

    Layout updates:
      - legend moved closer to the top panels
      - bottom heatmaps made slightly smaller
      - right heatmap y-axis labels removed to avoid overlap
    """
    _set_gb_rc(base_pt=9.2)
    spec = FigSpec()

    fig = plt.figure(figsize=(12.1, 5.2), dpi=300)

    gs = fig.add_gridspec(
        2, 4,
        height_ratios=[1.15, 0.78],
        hspace=0.52,
        wspace=0.38
    )

    fig.subplots_adjust(
        left=0.06,
        right=0.97,
        bottom=0.08,
        top=0.86
    )

    # -----------------------------
    # Top row: Fig. 3a
    # -----------------------------
    for i, ds_name in enumerate(DATASET_ORDER):
        ax = fig.add_subplot(gs[0, i])

        sub = summary_df[summary_df["Dataset"] == ds_name].copy()
        if sub.empty:
            ax.axis("off")
            continue

        sub = sub.set_index("Pipeline")
        x = np.arange(len(PIPELINE_ORDER))

        means = []
        err_lo = []
        err_hi = []
        present = []

        for p in PIPELINE_ORDER:
            if p in sub.index:
                m = float(sub.at[p, "MeanSanno_pct"])
                lo = float(sub.at[p, "MeanSanno_CI_Low_pct"])
                hi = float(sub.at[p, "MeanSanno_CI_High_pct"])
                means.append(m)
                err_lo.append(m - lo)
                err_hi.append(hi - m)
                present.append(True)
            else:
                means.append(np.nan)
                err_lo.append(0.0)
                err_hi.append(0.0)
                present.append(False)

        means = np.array(means, dtype=float)
        err = np.vstack([np.array(err_lo), np.array(err_hi)])
        present = np.array(present, dtype=bool)

        ax.bar(
            x[present],
            means[present],
            yerr=err[:, present],
            capsize=2.5,
            color=[PIPELINE_COLORS[p] for p in np.array(PIPELINE_ORDER)[present]],
            edgecolor="black",
            linewidth=0.5,
        )

        for xi, ok in zip(x, present):
            if not ok:
                ax.text(xi, 3, "NA", ha="center", va="bottom", fontsize=12)

        ax.set_xticks(x)
        ax.set_xticklabels([""] * len(x))
        ax.tick_params(axis="x", length=0)

        ax.set_ylim(0, 105)
        ax.set_yticks([0, 20, 40, 60, 80, 100])

        if i == 0:
            ax.set_ylabel("Mean S_anno (%)", fontweight="bold")
        else:
            ax.tick_params(axis="y", labelleft=False)

        ax.set_xlabel(DATASET_SPECS[ds_name]["display"], fontweight="bold", labelpad=2)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.grid(axis="y", linewidth=0.3)
        ax.set_axisbelow(True)

    # -----------------------------
    # Bottom left: Fig. 3b
    # -----------------------------
    ax_b = fig.add_subplot(gs[1, 0:2])
    draw_metric_heatmap(
        ax=ax_b,
        summary_df=summary_df,
        value_col="FracScoreGe0_5_pct",
        title="Ontology-consistent accuracy (%)\n(S_anno ≥ 0.5)",
        vmin=0,
        vmax=100,
        cmap="YlGnBu",
        show_ylabels=True,
    )

    # -----------------------------
    # Bottom right: Fig. 3c
    # -----------------------------
    ax_c = fig.add_subplot(gs[1, 2:4])
    im2 = draw_metric_heatmap(
        ax=ax_c,
        summary_df=summary_df,
        value_col="MajorLineageAcc_pct",
        title="Major-lineage accuracy (%)",
        vmin=0,
        vmax=100,
        cmap="YlGnBu",
        show_ylabels=True,  
    )

    cbar = fig.colorbar(im2, ax=[ax_b, ax_c], fraction=0.025, pad=0.02)
    cbar.set_label("Accuracy (%)", fontweight="bold")

    handles = [
        plt.matplotlib.patches.Patch(
            facecolor=PIPELINE_COLORS[p],
            edgecolor="black",
            linewidth=0.5,
            label=PIPELINE_LABELS[p],
        )
        for p in PIPELINE_ORDER
    ]

    fig.legend(
        handles=handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.96),
        ncol=len(PIPELINE_ORDER),
        frameon=False,
        handlelength=1.4,
        columnspacing=1.1,
        borderaxespad=0.2,
        fontsize=14
    )

    out_pdf = os.path.join(OUTPUT_DIR, "Fig3a_c.pdf")
    _save(fig, out_pdf, dpi_png=spec.DPI_PNG, pad_inches=spec.PAD_IN)
    plt.close(fig)
    print(f"[INFO] saved main Fig. 3 → {out_pdf}")

In [8]:
# ------------------------------------------------------------
# 6. Diagnostic heatmaps (not main figure)
# ------------------------------------------------------------
def plot_diagnostic_heatmaps(summary_df: pd.DataFrame, outdir: str):
    """
    Save diagnostic metrics separately:
      - exact-match accuracy
      - perfect ontology-aware agreement (Sanno == 1)
    """
    _set_gb_rc(base_pt=9.2)
    spec = FigSpec()

    fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.6), dpi=300)

    im1 = draw_metric_heatmap(
        ax=axes[0],
        summary_df=summary_df,
        value_col="ExactMatch_pct",
        title="Diagnostic: exact-match accuracy (%)",
        vmin=0,
        vmax=100,
        cmap="YlGnBu",
    )

    im2 = draw_metric_heatmap(
        ax=axes[1],
        summary_df=summary_df,
        value_col="FracScoreEq1_pct",
        title="Diagnostic: perfect ontology-aware agreement (%)\n(S_anno = 1.0)",
        vmin=0,
        vmax=100,
        cmap="YlGnBu",
    )

    cbar = fig.colorbar(im2, ax=axes, fraction=0.025, pad=0.03)
    cbar.set_label("Accuracy (%)")

    out_pdf = os.path.join(outdir, "Diagnostic_heatmaps.pdf")
    _save(fig, out_pdf, dpi_png=spec.DPI_PNG, pad_inches=spec.PAD_IN)
    plt.close(fig)
    print(f"[INFO] saved diagnostic heatmaps → {out_pdf}")


In [9]:
# ------------------------------------------------------------
# 7. Optional: save separate top-row mini-panels only
# ------------------------------------------------------------
def save_fig3_toprow_panels(summary_df: pd.DataFrame, outdir: str):
    _set_gb_rc(base_pt=10.0)
    spec = FigSpec()

    for ds_name in DATASET_ORDER:
        sub = summary_df[summary_df["Dataset"] == ds_name].copy()
        if sub.empty:
            continue

        sub = sub.set_index("Pipeline")
        x = np.arange(len(PIPELINE_ORDER))

        means = []
        err_lo = []
        err_hi = []
        present = []

        for p in PIPELINE_ORDER:
            if p in sub.index:
                m = float(sub.at[p, "MeanSanno_pct"])
                lo = float(sub.at[p, "MeanSanno_CI_Low_pct"])
                hi = float(sub.at[p, "MeanSanno_CI_High_pct"])
                means.append(m)
                err_lo.append(m - lo)
                err_hi.append(hi - m)
                present.append(True)
            else:
                means.append(np.nan)
                err_lo.append(0.0)
                err_hi.append(0.0)
                present.append(False)

        means = np.array(means, dtype=float)
        err = np.vstack([np.array(err_lo), np.array(err_hi)])
        present = np.array(present, dtype=bool)

        fig, ax = plt.subplots(figsize=(6.55, 2.35), dpi=300)
        ax.bar(
            x[present],
            means[present],
            yerr=err[:, present],
            capsize=2.5,
            color=[PIPELINE_COLORS[p] for p in np.array(PIPELINE_ORDER)[present]],
            edgecolor="black",
            linewidth=0.5,
        )

        for xi, ok in zip(x, present):
            if not ok:
                ax.text(xi, 3, "NA", ha="center", va="bottom", fontsize=8)

        ax.set_xticks(x)
        ax.set_xticklabels([""] * len(x))
        ax.tick_params(axis="x", length=0)

        ax.set_ylim(0, 105)
        ax.set_yticks([0, 20, 40, 60, 80, 100])

        if ds_name == "CD8":
            ax.set_ylabel("Mean Sanno (%)", fontweight="bold")
        else:
            ax.tick_params(axis="y", labelleft=False)

        ax.set_xlabel(DATASET_SPECS[ds_name]["display"], fontweight="bold", labelpad=2)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.grid(axis="y", linewidth=0.3)
        ax.set_axisbelow(True)

        out_pdf = os.path.join(OUTPUT_DIR, f"Figure3a_{ds_name}_panel.pdf")
        _save(fig, out_pdf, dpi_png=spec.DPI_PNG, pad_inches=spec.PAD_IN)
        plt.close(fig)

    handles = [
        plt.matplotlib.patches.Patch(
            facecolor=PIPELINE_COLORS[p],
            edgecolor="black",
            linewidth=0.5,
            label=PIPELINE_LABELS[p],
        )
        for p in PIPELINE_ORDER
    ]
    fig = plt.figure(figsize=(8.2, 0.85), dpi=300)
    ax = fig.add_subplot(111)
    ax.axis("off")
    fig.legend(
        handles=handles,
        loc="center",
        ncol=len(PIPELINE_ORDER),
        frameon=False,
        handlelength=1.4,
        columnspacing=1.2,
    )
    out_pdf = os.path.join(outdir, "Figure3a_legend.pdf")
    _save(fig, out_pdf, dpi_png=spec.DPI_PNG, pad_inches=spec.PAD_IN)
    plt.close(fig)

    print("[INFO] saved separate Fig. 3a mini-panels + legend")


In [10]:
# ------------------------------------------------------------
# 8. Extended Data confusion matrices
# ------------------------------------------------------------
PIPELINE_CMAPS = {
    "Standard": "Blues",
    "Curated": "Reds",
    "CellTypist": "Greens",
}

PIPELINE_TITLE_LABELS = {
    "Standard": "Standard",
    "Curated": "Full pipeline\n(LLM-scCurator)",
    "CellTypist": "CellTypist",
}

STATE_ORDER_MAP = {
    "CD8": ["Naive", "EffMem", "Exhausted", "Resident", "MAIT", "ISG", "Cycling", "Other"],
    "CD4": ["Naive", "EffMem", "Exhausted", "Treg", "Tfh", "Th17", "ISG", "Cycling", "Other"],
    "MSC": ["iCAF", "myCAF", "PVL", "Cycling", "Endothelial", "Other"],
    "MOUSE_B": ["Mature_B", "Erythrocyte_like", "Mast_like", "pDC_Myeloid_like", "Other"],
}

def compute_confusion_generic(
    df: pd.DataFrame,
    cfg,
    pipeline: str,
    state_order: list[str],
):
    pred_col = choose_prediction_column(df, pipeline)
    score_col = f"Score_{pipeline}"
    if score_col not in df.columns:
        raise ValueError(f"{score_col} not found.")

    gt_state_list = []
    used_list = []
    for gt in df["Ground_Truth"].astype(str):
        _, state = _expected_major_state_generic(gt, cfg)
        gt_state_list.append(state)
        used_list.append(state != cfg.default_state)

    tmp = df.copy()
    tmp["GT_State"] = gt_state_list
    tmp["UsedInConfusion"] = used_list

    df_used = tmp[tmp["UsedInConfusion"]].copy()

    labels = state_order[:]
    label_to_idx = {lab: i for i, lab in enumerate(labels)}

    cm_sum = np.zeros((len(labels), len(labels)), dtype=float)
    cm_count = np.zeros((len(labels), len(labels)), dtype=int)

    gt_states = []
    pred_states = []

    for _, row in df_used.iterrows():
        gt = str(row["GT_State"])
        pred = _parse_state_generic(str(row[pred_col]), cfg)
        score = float(row[score_col])

        gt_lab = gt if gt in label_to_idx else "Other"
        pred_lab = pred if pred in label_to_idx else "Other"

        i = label_to_idx[gt_lab]
        j = label_to_idx[pred_lab]

        cm_sum[i, j] += score
        cm_count[i, j] += 1

        gt_states.append(gt_lab)
        pred_states.append(pred_lab)

    with np.errstate(invalid="ignore", divide="ignore"):
        cm_mean = np.where(cm_count > 0, cm_sum / cm_count, 0.0)

    n = len(gt_states)
    n_correct = sum(1 for g, p in zip(gt_states, pred_states) if g == p)
    state_acc = 100.0 * n_correct / n if n > 0 else 0.0
    hier_mean = df_used[score_col].dropna().astype(float).mean() * 100.0

    return {
        "labels": labels,
        "cm_norm": cm_mean,
        "state_acc": state_acc,
        "n_correct": n_correct,
        "n": n,
        "hier_mean": hier_mean,
    }

def plot_extended_confusion(dataset_name: str, outdir: str):
    spec = DATASET_SPECS[dataset_name]
    csv_path = spec["csv"]
    if not os.path.exists(csv_path):
        print(f"[WARN] missing: {csv_path}")
        return

    df = pd.read_csv(csv_path)
    cfg = spec["cfg"]
    state_order = STATE_ORDER_MAP[dataset_name]
    pipelines = ["Standard", "Curated", "CellTypist"]

    results = {
        p: compute_confusion_generic(df, cfg, p, state_order)
        for p in pipelines
        if f"Score_{p}" in df.columns
    }

    fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 6), dpi=300)
    if len(results) == 1:
        axes = [axes]

    for ax, pipeline in zip(axes, results.keys()):
        res = results[pipeline]
        labels = res["labels"]
        cm = res["cm_norm"]

        ax.imshow(cm, vmin=0.0, vmax=1.0, cmap=PIPELINE_CMAPS[pipeline])

        ax.set_xticks(np.arange(len(labels)))
        ax.set_yticks(np.arange(len(labels)))
        ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=11)
        ax.set_yticklabels(labels, fontsize=11)

        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                val = cm[i, j]
                ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=10)

        title = (
            f"{PIPELINE_TITLE_LABELS[pipeline]}\n"
            f"State acc: {res['state_acc']:.1f}% ({res['n_correct']}/{res['n']})\n"
            f"Mean Sanno: {res['hier_mean']:.1f}%"
        )
        ax.set_title(
            title,
            fontsize=20,
            fontweight="bold",
            color=PIPELINE_COLORS[pipeline],
        )

        if ax is axes[0]:
            ax.set_ylabel("True state", fontsize=12, fontweight="bold")
        else:
            ax.set_yticklabels([])

        ax.set_xlabel("Predicted state", fontsize=12, fontweight="bold")

    plt.suptitle(f"{DATASET_SPECS[dataset_name]['display']} benchmark", fontsize=16, fontweight="bold", y=1.03)
    plt.tight_layout()

    out_pdf = os.path.join(OUTPUT_DIR, f"FigS4_{dataset_name}_confusion.pdf")
    _save(fig, out_pdf, dpi_png=600, pad_inches=0.03)
    plt.close(fig)
    print(f"[INFO] saved confusion figure → {out_pdf}")


In [11]:
# ------------------------------------------------------------
# 9. Run
# ------------------------------------------------------------
summary_df = build_fig3_summary()
summary_csv = os.path.join(OUTPUT_DIR, "Fig3_data.csv")
summary_df.to_csv(summary_csv, index=False)
print(f"[INFO] saved summary table → {summary_csv}")
print(summary_df)

print_debug_tables()

plot_fig3_main(summary_df, OUTPUT_DIR)
plot_diagnostic_heatmaps(summary_df, OUTPUT_DIR)
save_fig3_toprow_panels(summary_df, OUTPUT_DIR)

for ds_name in DATASET_ORDER:
    plot_extended_confusion(ds_name, OUTPUT_DIR)

[INFO] saved summary table → /work/paper/gb_resubmission/output/Fig3/Fig3_data.csv
    Dataset    Pipeline   PredictionColumn  N_clusters  N_eval  MeanSanno  \
0       CD8    Standard    Standard_Answer          17      17   0.752941   
1       CD8     Curated     Curated_Answer          17      17   0.817647   
2       CD8  CellTypist  CellTypist_Answer          17      17   0.735294   
3       CD8     SingleR     SingleR_Answer          17      17   0.858824   
4       CD8     Azimuth     Azimuth_Answer          17      17   0.835294   
5       CD4    Standard    Standard_Answer          22      22   0.750000   
6       CD4     Curated     Curated_Answer          22      22   0.850000   
7       CD4  CellTypist  CellTypist_Answer          22      22   0.863636   
8       CD4     SingleR     SingleR_Answer          22      22   0.604545   
9       CD4     Azimuth     Azimuth_Answer          22      22   0.863636   
10      MSC    Standard    Standard_Answer           8       8   0.806